In [ ]:
import tqdm

In [ ]:
import re

In [ ]:
import json

In [ ]:
import pickle

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

False

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [ ]:
with open("tags_set.pkl", "rb") as f:
    tags_set = pickle.load(f)

In [ ]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [ ]:
len(titles_with_tags)

1164

In [ ]:
data = []

In [ ]:
for title, tags in titles_with_tags.items():
    for tag in tags:
        data.append((title, tag))

In [ ]:
data[0]

('Исследование приоритетов и механизмов реализации отраслевых (секторальных) политик в избранных странах БРИКС (членах и кандидатах), во взаимосвязи с повесткой международных доноров',
 'international_relations')

In [ ]:
data[-1]

('Обзор Методологии в Области Регионоведения. Этап сбора данных 1.',
 'geography')

In [ ]:
df = pd.DataFrame(data=data, columns=["title_rus", "tag"])

In [ ]:
df

,title_rus,tag
0,Исследование приоритетов и механизмов реализац...,international_relations
1,Исследование приоритетов и механизмов реализац...,political_economics
2,Исследование приоритетов и механизмов реализац...,policy_analysis
3,Исследование приоритетов и механизмов реализац...,BRICS
4,Исследование приоритетов и механизмов реализац...,geopolitics
...,...,...
5977,Актив Центра развития карьеры - профориентацио...,career_planning
5978,Обзор Методологии в Области Регионоведения. Эт...,regional_studies
5979,Обзор Методологии в Области Регионоведения. Эт...,methodology
5980,Обзор Методологии в Области Регионоведения. Эт...,data_collection


In [ ]:
dfgr = df.groupby(by=["tag"]).agg({"title_rus": "nunique"}).reset_index()
dfgr.columns = ["tag", "count"]
dfgr = dfgr.sort_values(by=["count"], ascending=False).reset_index(drop=True)
dfgr.head(32)

,tag,count
0,economics,96
1,international_relations,94
2,linguistics,91
3,education,86
4,sociology,86
5,marketing,85
6,cultural_studies,82
7,software_engineering,79
8,history,73
9,media_studies,69


In [ ]:
selected_tags = dfgr.head(32)["tag"].to_list()

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can create artificial user profiles based on student project titles in russian.
Please, analyze given list of student project titles in russian and create one possible user profile with its own preferences in a provided field of science.
Also select several relevant tags from a tags set.
Return stricly JSON output, consistin of user name, brief user bio and relevant tags list.
For example.
Input data:
```json
{"student_project_titles": ["Исследование методов решения задачи линейной регрессии", "Исследование методов решения задачи линейной классификации", "Исследование методов решения задачи обучения с подкреплением"], "all_possible_tags": ["machine_learning", "computer_science", "economics", "politics"]}
```
Your output:
```json
{"machine_learning_researcher_classic_algorithms": {"bio": "A user, which is interested in machine learning and classic algorithms in particular", "tags": ["machine_learning", "computer_science"]}}
```
"""

In [ ]:
messages = []

In [ ]:
for tag in selected_tags:
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {"student_project_titles": list(df[df["tag"] == tag]["title_rus"].unique()), "all_possible_tags": list(tags_set)},
                    ensure_ascii=False,
                ),
            },
        )
    )

In [ ]:
answers = {}

In [ ]:
for tag, message in tqdm.tqdm(list(zip(selected_tags, messages))):
    try:
        text = processor.apply_chat_template(
            message, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )

        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(**inputs, max_new_tokens=4096)
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

        json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_str = response

        answers.update(json.loads(json_str))
    except Exception as e:
        print(f"something went wrong with tag {tag} - {e}")

 72%|███████▏  | 23/32 [05:22<01:52, 12.55s/it]

something went wrong with tag project_management - Expecting ',' delimiter: line 15 column 2 (char 562)


100%|██████████| 32/32 [07:29<00:00, 14.06s/it]


In [ ]:
len(answers)

31

In [ ]:
answers[list(answers.keys())[0]]

{'bio': 'A highly analytical researcher deeply engaged in monitoring global economic trends, geopolitical risks, international trade, and the socio-economic development of various regions (especially BRICS, Asia, and the Middle East). Interested in the intersection of macroeconomics, international relations, and the impact of global events on business and society.',
 'tags': ['global_economy',
  'geopolitics',
  'macroeconomics',
  'international_economics',
  'data_analysis',
  'forex',
  'regional_development',
  'brics',
  'political_economy',
  'international_relations',
  'data_monitoring',
  'time_series',
  'system_thinking',
  'geopolitics_of_BRICS',
  'investment_analysis',
  'economic_forecasting',
  'trade_policy',
  'market_forecasting',
  'data_mining',
  'quantitative_finance',
  'global_systems',
  'regional_analysis',
  'economic_development']}

In [ ]:
answers[list(answers.keys())[-1]]

{'bio': 'A passionate researcher with a deep interest in cultural studies, media production, and cross-cultural communication, particularly focusing on Eastern and Middle Eastern cultures. Interested in how narratives, visual arts, and digital platforms shape societal values.',
 'tags': ['cultural_studies',
  'media_studies',
  'visual_communication',
  'cross_cultural_communication',
  'film_studies',
  'narrative_writing',
  'content_creation',
  'Middle_East_studies',
  'international_communication',
  'media_production']}

In [ ]:
with open("artificial_profiles.json", "w") as f:
    json.dump(answers, f)